In [6]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import torch.nn as nn
import matplotlib.pyplot as plt

### Preprocessing 

In [7]:
df = pd.read_csv('./Reddit_Data.csv')

In [8]:

print(df.info())
print("Target data distribution: ",[int((i/37249)*100) for i in df.category.value_counts()])


df.dropna(inplace=True)
print("Total null Values: ",df.isnull().sum())
#lower casing----------------------------
df['clean_comment'] = df['clean_comment'].str.lower()

#remove punctuation----------------------------
import string
exclude = string.punctuation
print(exclude)
def removepunc(text:str):
    return text.translate(str.maketrans('','',exclude))

df['clean_comment'] = df['clean_comment'].apply(removepunc)
df.head()

#stop word removal----------------------------
# !pip install nltk
import nltk
from nltk.corpus import stopwords
import time
# nltk.download('stopwords')
en_stopwords = stopwords.words('english')

def removestopwords(text:str):
    newword = []
    for word in text.split():
        if word not in en_stopwords:
            newword.append(word)
    x = newword[:]
    newword.clear()
    return " ".join(x)

df['clean_comment'] = df.clean_comment.apply(removestopwords)

#emoji replacement with their meaning----------------------------
# !pip install emoji
import emoji
df.clean_comment = df.clean_comment.apply(emoji.demojize)


#Nltk not that accurate but for now, just use it----------------------------
from nltk.tokenize import word_tokenize
# nltk.download('punkt_tab')
df.clean_comment = df.clean_comment.apply(word_tokenize)
df.head()

from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
def stem_word(text_row):
    return (" ".join(ps.stem(word) for word in text_row))
df.clean_comment = df.clean_comment.apply(stem_word)
df['split_comment'] = df.clean_comment.str.split()

df.head(1)



<class 'pandas.DataFrame'>
RangeIndex: 37249 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   clean_comment  37149 non-null  str  
 1   category       37249 non-null  int64
dtypes: int64(1), str(1)
memory usage: 7.0 MB
None
Target data distribution:  [42, 35, 22]
Total null Values:  clean_comment    0
category         0
dtype: int64
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


,clean_comment,category,split_comment
0,famili mormon never tri explain still stare pu...,1,"[famili, mormon, never, tri, explain, still, s..."


In [9]:
x = df['split_comment']
y = df['category']
x_train, x_test, y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)


print(x_train.shape)
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
x_train.head(1)

(29719,)
x_train: (29719,)
x_test: (7430,)


31732    [oooooooooooh, rememb, chutiya, one, shout, ma...
Name: split_comment, dtype: object

### Load trained BERT representatioin

In [10]:
# cls = torch.load('./weights/cls_representation_final.pt')
cls = torch.load('./embeddings.pt')
cls = cls.sum(axis=1)
print(cls.shape)
# print(cls2[0,0,:])

torch.Size([29719, 96])


In [11]:
mean = cls.mean(dim=-1, keepdim=True)
std = cls.std(dim=-1, keepdim=True).clamp_min(1e-8)
cls = (cls - mean) / std
cls


tensor([[ 2.1467, -1.2142,  0.5382,  ...,  0.7276, -0.4616, -1.8109],
        [ 1.9953, -1.3808,  0.5213,  ...,  0.7656, -0.2073, -1.7032],
        [ 1.9597, -1.3949,  0.5331,  ...,  0.7679, -0.1693, -1.6908],
        ...,
        [ 1.5260,  1.2475,  0.1120,  ..., -0.1668, -2.4099, -1.0878],
        [ 1.9130, -1.4548,  0.5088,  ...,  0.8040, -0.1095, -1.6631],
        [ 1.5892,  1.1750,  0.2041,  ..., -0.1964, -2.2252, -1.1328]],
       device='cuda:0')

In [12]:
train_idx = x_train.index.to_numpy()
final_dataset = pd.DataFrame({
    'tokens': x_train.values,
    'token_bert': cls.detach().cpu().tolist(),
    'sentiment': y_train.values
})


In [13]:
final_dataset.head(5)

,tokens,token_bert,sentiment
0,"[oooooooooooh, rememb, chutiya, one, shout, ma...","[2.146704912185669, -1.2142152786254883, 0.538...",1
1,"[one, good, hijack, media]","[1.9952938556671143, -1.3807779550552368, 0.52...",1
2,"[harshvardhan, ahead, 43350, vote]","[1.9596517086029053, -1.39493989944458, 0.5330...",0
3,"[epap, anyon, want, free, encyclopedia, team, ...","[2.1351158618927, 0.5601490139961243, 0.376324...",1
4,"[fuck, boy, heart, pound]","[1.965203046798706, -1.399353265762329, 0.5215...",-1


In [14]:
a = np.asarray(final_dataset.token_bert[12], dtype=float)
b = np.asarray(final_dataset.token_bert[19], dtype=float)
cosine_similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
cosine_similarity

np.float64(0.992594041904125)

In [15]:
from typing import Literal

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("cuda is available")
else:
    print("cuda is not available")
    
class LegacyMLP(nn.Module):
  def __init__(self, inp_Dense_out=[]):
      super().__init__()
      layers = []
      for i in range(len(inp_Dense_out) - 2):
          layers.append(nn.Linear(inp_Dense_out[i], inp_Dense_out[i + 1]))
          layers.append(nn.ReLU())

      layers.append(nn.Linear(inp_Dense_out[-2], inp_Dense_out[-1]))
      # layers.append(nn.Softmax())
      self.model = nn.Sequential(*layers)
      self.model = self.model.to(device)

  def parameters(self):
      return self.model.parameters()

  def forward(self, x):
    #   x = torch.dropout(x, p=0.2, train=self.training)
      return self.model(x)

  def Fit(self,x,y,epoch=1,batch_size=1,lr=0.1, loss_fn: Literal['mse','cross_entropy']='cross_entropy'):
      # loss_fn = nn.CrossEntropyLoss()
      loss_fn = nn.MSELoss() if loss_fn == 'mse' else nn.CrossEntropyLoss()
    # loss_fn = nn.
      optimizer = torch.optim.AdamW(self.parameters(), lr=lr)
      epoch_loss_history = []
      for epoch in range(epoch):
          epoch_loss = 0
          for i in range(0, len(x), batch_size):
              x_batch = x[i:i+batch_size]
              y_batch = y[i:i+batch_size]
              #Forward
              y_pred = self.forward(x_batch)
              #Grad reset
              optimizer.zero_grad()
              #loss
              loss = loss_fn(y_pred, y_batch)
            #   print("loss", loss)
              #Backward Grad calculation
              loss.backward()
              #Peramater adjustment
              optimizer.step()
              #loss register for the batch
              epoch_loss+=loss.item()
          epoch_loss = epoch_loss/(len(x)/batch_size)
          epoch_loss_history.append(loss)
      return epoch_loss_history

  def predict(self, x):
    with torch.no_grad():
        x = x.to(device)     # Ensure your input tensor is on the same device
        outputs = model(x)     # Forward pass to get raw model outputs (logits)
        predictions = torch.argmax(outputs, dim=1)      # Post-process outputs (e.g., get class predictions)
    return predictions

  def save(self):
    torch.save(self.state_dict(), '../weights/mlp')

  def load(self):
    self.load_state_dict(torch.load('../weights/mlp'))

from mlp import MLP


cuda is available


In [16]:
x = np.stack(final_dataset.token_bert[:], dtype=float)
y = np.stack(final_dataset.sentiment[:],dtype=float)

x.shape,y.shape

((29719, 96), (29719,))

In [17]:
features = final_dataset.token_bert.shape[-1]

x_tensor = torch.as_tensor(x, dtype=torch.float32, device=device)
y_tensor = torch.as_tensor(y, dtype=torch.float32, device=device).reshape(-1, 1)
#====================================
# CrossEntropyLoss requires:
# - target shape: [batch_size]
# - target dtype: torch.long
# - class labels: 0, 1, ..., num_classes - 1
# Convert labels from {-1, 0, 1} to {0, 1, 2}.
y_tensor = torch.as_tensor(
    y + 1,
    dtype=torch.long,
    device=device
)
x_train, x_test, y_train,y_test = train_test_split(x_tensor,y_tensor,test_size=0.2,random_state=42)

# Recreate the model with the actual embedding dimension
model = MLP([96, 32, 32, 16, 3])
model = model.to(device)

print(x_train.shape,y_train.shape)



torch.Size([23775, 96]) torch.Size([23775])


In [ ]:
lossH = model.Fit(
    x_train[:],
    y_train[:],
    epoch=50,
    lr=0.01,
    batch_size=10,
    loss_fn="cross_entropy"
)
plt.plot(lossH)


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

y_pred = model.predict(x=x_test)
accuracy = confusion_matrix(y_test.cpu(), y_pred.cpu())
accuracy
# y_pred

In [27]:
%pip install -q "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.


In [31]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Scale data (X contains your numeric features)
scaler = StandardScaler()

X_scaled = scaler.fit_transform(x_train.detach().cpu().numpy())

# 2. Reduce to 2 dimensions
pca = PCA(n_components=1)
X_pca = pca.fit_transform(X_scaled)

# 3. Plot 2D PCA
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train.detach().cpu().numpy(), cmap="viridis", edgecolor="k")
plt.xlabel("Principal Component 1 (PC1)")
plt.ylabel("Principal Component 2 (PC2)")
plt.title("2D PCA Plot")
plt.colorbar(label="Classes")
plt.show()


IndexError: index 1 is out of bounds for axis 1 with size 1

<Figure size 800x600 with 0 Axes>